# RoBERTa Stance Detection — FNC-1

Fine-tuning `roberta-base` for the Fake News Challenge (FNC-1) stance detection task.

**Purpose:** Baseline comparison against Longformer (`allenai/longformer-base-4096`).

**Key features:**
- Leakage-free data splitting using `GroupShuffleSplit` on Body ID
- Tokenization analysis for optimal MAX_LENGTH selection
- Class weight analysis for handling class imbalance
- Comprehensive evaluation with error analysis

# Environment Configuration

In [ ]:
# Environment Configuration & Setup
ENVIRONMENT = "kaggle"
# ENVIRONMENT = "local"

if ENVIRONMENT == "kaggle":
    TRAIN_PATH = "/kaggle/input/datasets/deanfebrio/aol-textmining-fakenewsdetection-dataset/processed/train_processed.csv"
    TEST_PATH = "/kaggle/input/datasets/deanfebrio/aol-textmining-fakenewsdetection-dataset/processed/test_processed.csv"
    OUTPUT_DIR = "/kaggle/working/model-roberta/"
elif ENVIRONMENT == "local":
    TRAIN_PATH = "./data/processed/train_processed.csv"
    TEST_PATH = "./data/processed/test_processed.csv"
    OUTPUT_DIR = "./artifacts/model-roberta/"
else:
    raise ValueError(f"Unknown environment: {ENVIRONMENT}")

print(f"Environment: {ENVIRONMENT}")
print(f"Train path: {TRAIN_PATH}")
print(f"Test path: {TEST_PATH}")
print(f"Output dir: {OUTPUT_DIR}")

In [ ]:
if ENVIRONMENT == "kaggle":
    !pip install -q pandas numpy matplotlib seaborn scikit-learn torch transformers datasets accelerate tqdm

# Setup

Import library, konfigurasi random seed, dan deteksi device.

In [ ]:
import pandas as pd
import numpy as np
import os

import torch
import transformers

from transformers import (
    RobertaTokenizerFast,
    RobertaForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
)
from datasets import Dataset
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
)
from sklearn.utils.class_weight import compute_class_weight
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import time
import json
import warnings

warnings.filterwarnings("ignore")

# Random seed for reproducibility
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# Device detection
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU name: {torch.cuda.get_device_name(0)}")
    print(f"GPU memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")
print(f"Torch version: {torch.__version__}")
print(f"Transformers version: {transformers.__version__}")
print(f"Device: {device}")

# Load Dataset

Memuat dataset training dan testing yang sudah dipreprocessing.

In [ ]:
df_train = pd.read_csv(TRAIN_PATH)
df_test = pd.read_csv(TEST_PATH)

print("=== Train Dataset ===")
print(f"Shape: {df_train.shape}")
print(f"Columns: {df_train.columns.tolist()}")

print("\n=== Test Dataset ===")
print(f"Shape: {df_test.shape}")
print(f"Columns: {df_test.columns.tolist()}")

# Verify required columns
for col in ["Headline", "articleBody", "Stance", "Body ID"]:
    assert col in df_train.columns, f"Missing column in train: {col}"
    print(f"Column '{col}' verified in train dataset")

for col in ["Headline", "articleBody"]:
    assert col in df_test.columns, f"Missing column in test: {col}"
    print(f"Column '{col}' verified in test dataset")

print("\n=== Label Distribution (Train) ===")
print(df_train["Stance"].value_counts())

print(f"\nUnique Body IDs in train: {df_train['Body ID'].nunique()}")

# Label Encoding

Mapping label stance ke nilai numerik untuk training.

In [ ]:
label2id = {
    "agree": 0,
    "disagree": 1,
    "discuss": 2,
    "unrelated": 3,
}

id2label = {v: k for k, v in label2id.items()}

print("label2id:", label2id)
print("id2label:", id2label)

# Convert labels to numeric
df_train["label"] = df_train["Stance"].map(label2id)

print("\n=== Label Counts ===")
print(df_train["label"].value_counts().sort_index())

# Verify no NaN labels
assert df_train["label"].isna().sum() == 0, "Found unmapped labels!"
print("\nAll labels mapped successfully.")

# Train Validation Split (Leakage-Free)

**CRITICAL:** Menggunakan `GroupShuffleSplit` dengan `Body ID` sebagai grouping variable.

Ini memastikan bahwa **tidak ada Body ID yang muncul di kedua train dan validation set**, identik dengan strategi yang digunakan pada Longformer notebook.

In [ ]:
# =====================================================
# Group-Based Split using Body ID
# =====================================================
# GroupShuffleSplit ensures that all rows sharing the same
# Body ID are placed entirely in either train or validation,
# never split across both.
# =====================================================

gss = GroupShuffleSplit(n_splits=1, test_size=0.1, random_state=SEED)

groups = df_train["Body ID"].values

train_idx, val_idx = next(gss.split(X=df_train, y=df_train["label"], groups=groups))

df_train_split = df_train.iloc[train_idx].reset_index(drop=True)
df_val_split = df_train.iloc[val_idx].reset_index(drop=True)

print(f"Train split size: {len(df_train_split)}")
print(f"Validation split size: {len(df_val_split)}")
print(f"\nTrain unique Body IDs: {df_train_split['Body ID'].nunique()}")
print(f"Validation unique Body IDs: {df_val_split['Body ID'].nunique()}")

# Leakage Verification

Verifikasi bahwa tidak ada Body ID yang muncul di kedua train dan validation set. Overlap count harus **0**.

In [ ]:
# =====================================================
# Leakage Verification
# =====================================================

train_body_ids = set(df_train_split["Body ID"].unique())
val_body_ids = set(df_val_split["Body ID"].unique())
overlap_body_ids = train_body_ids & val_body_ids

print(f"Number of unique train Body IDs:      {len(train_body_ids)}")
print(f"Number of unique validation Body IDs:  {len(val_body_ids)}")
print(f"Number of overlapping Body IDs:        {len(overlap_body_ids)}")

if len(overlap_body_ids) > 0:
    import warnings
    warnings.warn(
        f"⚠️ DATA LEAKAGE DETECTED! {len(overlap_body_ids)} Body IDs appear in both "
        f"train and validation sets. This will inflate validation metrics.",
        UserWarning,
    )
    print(f"\n⚠️ WARNING: Overlapping Body IDs: {sorted(list(overlap_body_ids))[:20]}...")
else:
    print("\n✅ No data leakage detected. Train and validation Body IDs are completely disjoint.")

# Distribution Analysis

Distribusi label pada train dan validation split setelah group-based splitting.

In [ ]:
print("=== Train Split Distribution ===")
print(df_train_split["Stance"].value_counts())
print(f"\nPercentages:")
print((df_train_split["Stance"].value_counts(normalize=True) * 100).round(2))

print("\n=== Validation Split Distribution ===")
print(df_val_split["Stance"].value_counts())
print(f"\nPercentages:")
print((df_val_split["Stance"].value_counts(normalize=True) * 100).round(2))

# Visualization
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

order = ["agree", "disagree", "discuss", "unrelated"]

sns.countplot(data=df_train_split, x="Stance", order=order, ax=axes[0], palette="viridis")
axes[0].set_title("Train Split — Label Distribution", fontsize=13, fontweight="bold")
axes[0].set_xlabel("Stance")
axes[0].set_ylabel("Count")
for container in axes[0].containers:
    axes[0].bar_label(container, fmt="%d")

sns.countplot(data=df_val_split, x="Stance", order=order, ax=axes[1], palette="viridis")
axes[1].set_title("Validation Split — Label Distribution", fontsize=13, fontweight="bold")
axes[1].set_xlabel("Stance")
axes[1].set_ylabel("Count")
for container in axes[1].containers:
    axes[1].bar_label(container, fmt="%d")

plt.tight_layout()
plt.show()

# RoBERTa Initialization

Load tokenizer dan model dari `roberta-base` dengan konfigurasi 4 label untuk multi-class classification.

In [ ]:
MODEL_NAME = "roberta-base"

tokenizer = RobertaTokenizerFast.from_pretrained(MODEL_NAME)

model = RobertaForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=4,
    label2id=label2id,
    id2label=id2label,
)

model.to(device)

print(f"Model: {MODEL_NAME}")
print(f"Num labels: {model.config.num_labels}")
print(f"Max position embeddings: {model.config.max_position_embeddings}")
print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"Trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

# Tokenization Analysis

Analisis panjang token menggunakan RoBERTa tokenizer untuk menentukan `MAX_LENGTH` yang optimal.

**Catatan penting:** RoBERTa memiliki batas maksimum 512 token, berbeda dengan Longformer yang mendukung hingga 4096 token. Artikel yang lebih panjang dari 512 token akan di-truncate.

In [ ]:
# =====================================================
# Token Length Analysis
# =====================================================

def count_tokens(text):
    """Count tokens for a given text using the loaded tokenizer."""
    return len(tokenizer.encode(str(text), add_special_tokens=False))

print("Calculating token lengths (this may take a moment)...")

headline_tokens = df_train_split["Headline"].apply(count_tokens)
article_tokens = df_train_split["articleBody"].apply(count_tokens)
# Combined: <s> headline </s></s> article </s> = headline + article + 4 special tokens
combined_tokens = headline_tokens + article_tokens + 4

def report_token_stats(series, name):
    print(f"\n--- {name} ---")
    print(f"Mean:   {series.mean():.1f}")
    print(f"Median: {series.median():.1f}")
    print(f"Max:    {series.max()}")
    print(f"P95:    {np.percentile(series, 95):.1f}")
    print(f"P99:    {np.percentile(series, 99):.1f}")

report_token_stats(headline_tokens, "Headline Token Length")
report_token_stats(article_tokens, "Article Token Length")
report_token_stats(combined_tokens, "Combined Token Length")

# Coverage analysis for different MAX_LENGTH values
print("\n--- Coverage Analysis ---")
for max_len in [128, 256, 384, 512]:
    coverage = (combined_tokens <= max_len).mean() * 100
    print(f"MAX_LENGTH={max_len}: {coverage:.2f}% of samples fit without truncation")

In [ ]:
# =====================================================
# Token Length Visualizations
# =====================================================

fig, axes = plt.subplots(2, 3, figsize=(18, 10))

# Histograms
sns.histplot(headline_tokens, bins=30, kde=True, ax=axes[0, 0], color="#2196F3")
axes[0, 0].set_title("Headline Token Length — Histogram", fontweight="bold")
axes[0, 0].set_xlabel("Token Count")

sns.histplot(article_tokens, bins=50, kde=True, ax=axes[0, 1], color="#F44336")
axes[0, 1].set_title("Article Token Length — Histogram", fontweight="bold")
axes[0, 1].set_xlabel("Token Count")

sns.histplot(combined_tokens, bins=50, kde=True, ax=axes[0, 2], color="#4CAF50")
axes[0, 2].axvline(x=512, color="red", linestyle="--", label="MAX_LENGTH=512")
axes[0, 2].legend()
axes[0, 2].set_title("Combined Token Length — Histogram", fontweight="bold")
axes[0, 2].set_xlabel("Token Count")

# Boxplots
sns.boxplot(y=headline_tokens, ax=axes[1, 0], color="#2196F3")
axes[1, 0].set_title("Headline Token Length — Boxplot", fontweight="bold")
axes[1, 0].set_ylabel("Token Count")

sns.boxplot(y=article_tokens, ax=axes[1, 1], color="#F44336")
axes[1, 1].set_title("Article Token Length — Boxplot", fontweight="bold")
axes[1, 1].set_ylabel("Token Count")

sns.boxplot(data=pd.DataFrame({
    "Headline": headline_tokens,
    "Article": article_tokens,
    "Combined": combined_tokens,
}), ax=axes[1, 2])
axes[1, 2].set_title("Token Length Comparison — Boxplot", fontweight="bold")
axes[1, 2].set_ylabel("Token Count")

plt.tight_layout()
plt.show()

# RoBERTa Tokenization

Tokenisasi headline dan article body menggunakan RoBERTa tokenizer.

**MAX_LENGTH:** Configurable, default 512 (batas maksimum RoBERTa). Headline dan article body digabungkan sebagai input pair.

In [ ]:
# =====================================================
# Configurable Max Length
# =====================================================
MAX_LENGTH = 512

print(f"MAX_LENGTH: {MAX_LENGTH}")
print(f"Percentage of combined tokens within MAX_LENGTH: "
      f"{(combined_tokens <= MAX_LENGTH).mean() * 100:.2f}%")

In [ ]:
def tokenize_data(df, has_labels=True):
    """
    Tokenize dataframe using RoBERTa tokenizer.

    Args:
        df: DataFrame with 'Headline' and 'articleBody' columns
        has_labels: Whether the dataframe has 'label' column

    Returns:
        dict with input_ids, attention_mask, and optionally labels
    """
    # Tokenize headline + article as text pair
    encodings = tokenizer(
        df["Headline"].astype(str).tolist(),
        df["articleBody"].astype(str).tolist(),
        padding="max_length",
        truncation=True,
        max_length=MAX_LENGTH,
        return_tensors=None,
    )

    result = {
        "input_ids": encodings["input_ids"],
        "attention_mask": encodings["attention_mask"],
    }

    if has_labels:
        result["labels"] = df["label"].tolist()

    return result

print("Tokenizing train split...")
train_encodings = tokenize_data(df_train_split, has_labels=True)

print("Tokenizing validation split...")
val_encodings = tokenize_data(df_val_split, has_labels=True)

print("Tokenizing test set...")
test_encodings = tokenize_data(df_test, has_labels=False)

print("\nTokenization complete.")
print(f"Train samples: {len(train_encodings['input_ids'])}")
print(f"Validation samples: {len(val_encodings['input_ids'])}")
print(f"Test samples: {len(test_encodings['input_ids'])}")

# Hugging Face Dataset

Konversi hasil tokenisasi ke format Hugging Face Dataset dan set format PyTorch.

In [ ]:
train_dataset = Dataset.from_dict(train_encodings)
val_dataset = Dataset.from_dict(val_encodings)
test_dataset = Dataset.from_dict(test_encodings)

# Set PyTorch format
train_dataset.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "labels"],
)
val_dataset.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "labels"],
)
test_dataset.set_format(
    type="torch",
    columns=["input_ids", "attention_mask"],
)

print(f"Train dataset: {train_dataset}")
print(f"Validation dataset: {val_dataset}")
print(f"Test dataset: {test_dataset}")

# Class Weight Analysis

Menghitung bobot kelas berdasarkan distribusi label di training set. Bobot ini dapat digunakan untuk menangani class imbalance.

In [ ]:
# =====================================================
# Class Weight Analysis
# =====================================================

train_labels = np.array(df_train_split["label"].tolist())

class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(train_labels),
    y=train_labels,
)

class_weight_dict = {i: w for i, w in enumerate(class_weights)}

print("=== Class Frequencies ===")
for label_id in sorted(id2label.keys()):
    count = (train_labels == label_id).sum()
    pct = count / len(train_labels) * 100
    print(f"  {id2label[label_id]:>10s} (id={label_id}): {count:>6d} ({pct:.2f}%)")

print("\n=== Class Weights ===")
for label_id in sorted(id2label.keys()):
    print(f"  {id2label[label_id]:>10s} (id={label_id}): {class_weights[label_id]:.4f}")

# Store as tensor for potential custom loss usage
class_weights_tensor = torch.tensor(class_weights, dtype=torch.float32).to(device)
print(f"\nClass weights tensor: {class_weights_tensor}")

# Metrics Function

Fungsi `compute_metrics()` untuk evaluasi model selama training.

In [ ]:
def compute_metrics(eval_pred):
    """
    Compute evaluation metrics for the Trainer.

    Args:
        eval_pred: EvalPrediction object with predictions and label_ids

    Returns:
        dict with accuracy, precision_macro, recall_macro, f1_macro, f1_weighted
    """
    predictions, labels = eval_pred
    preds = np.argmax(predictions, axis=-1)

    return {
        "accuracy": accuracy_score(labels, preds),
        "precision_macro": precision_score(labels, preds, average="macro", zero_division=0),
        "recall_macro": recall_score(labels, preds, average="macro", zero_division=0),
        "f1_macro": f1_score(labels, preds, average="macro", zero_division=0),
        "f1_weighted": f1_score(labels, preds, average="weighted", zero_division=0),
    }

print("compute_metrics() defined.")

# Training Configuration

Konfigurasi `TrainingArguments` untuk Hugging Face Trainer.

In [ ]:
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=8,
    learning_rate=2e-5,
    weight_decay=0.01,
    warmup_ratio=0.1,
    num_train_epochs=5,
    fp16=True,
    gradient_checkpointing=True,
    eval_accumulation_steps=4,
    gradient_accumulation_steps=4,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,
    logging_steps=50,
    save_total_limit=2,
    seed=SEED,
    report_to="none",
    dataloader_num_workers=0,
    remove_unused_columns=False,
)

print("=== Training Configuration ===")
print(f"Batch size (per device): {training_args.per_device_train_batch_size}")
print(f"Learning rate: {training_args.learning_rate}")
print(f"Weight decay: {training_args.weight_decay}")
print(f"Warmup ratio: {training_args.warmup_ratio}")
print(f"Epochs: {training_args.num_train_epochs}")
print(f"FP16: {training_args.fp16}")
print(f"Eval strategy: {training_args.eval_strategy}")
print(f"Best model metric: {training_args.metric_for_best_model}")

# Early Stopping

Callback untuk menghentikan training lebih awal jika metrik evaluasi tidak membaik setelah beberapa epoch.

In [ ]:
early_stopping = EarlyStoppingCallback(
    early_stopping_patience=2,
    early_stopping_threshold=0.0,
)

print(f"Early stopping patience: {early_stopping.early_stopping_patience}")

# Trainer Initialization

Inisialisasi Hugging Face Trainer dengan model, tokenizer, dataset, metrics, dan callbacks.

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
    callbacks=[early_stopping],
)

print("Trainer initialized successfully.")
print(f"Train dataset size: {len(train_dataset)}")
print(f"Eval dataset size: {len(val_dataset)}")

# Training

Eksekusi training model RoBERTa. Proses ini akan:
1. Training selama beberapa epoch sesuai konfigurasi
2. Evaluasi di setiap akhir epoch
3. Menyimpan checkpoint terbaik berdasarkan F1 macro
4. Menghentikan training lebih awal jika tidak ada peningkatan (early stopping)

In [ ]:
start_time = time.time()

import os
from transformers.trainer_utils import get_last_checkpoint

# Cek apakah ada checkpoint sebelumnya di folder output
last_checkpoint = None
if os.path.isdir(OUTPUT_DIR):
    last_checkpoint = get_last_checkpoint(OUTPUT_DIR)

if last_checkpoint is not None:
    print(f"Melanjutkan training dari checkpoint: {last_checkpoint}")
    train_result = trainer.train(resume_from_checkpoint=last_checkpoint)
else:
    print("Memulai training dari awal...")
    train_result = trainer.train()

training_time = time.time() - start_time
print(f"\nTraining completed in {training_time / 60:.2f} minutes")
print(f"\n=== Training Results ===")
for key, value in train_result.metrics.items():
    print(f"  {key}: {value}")

# Training Curves

Visualisasi training history dari Trainer untuk mendeteksi overfitting secara visual.

Plot yang dihasilkan:
1. **Training Loss vs Epoch**
2. **Validation Loss vs Epoch**
3. **Validation F1 Macro vs Epoch**

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Set tema visual grafis
sns.set_theme(style="whitegrid")

# 1. Ekstrak data historis dari objek Hugging Face Trainer
history = trainer.state.log_history

train_loss, val_loss = [], []
train_acc, val_acc = [], []
epochs_loss, epochs_acc = [], []

for log in history:
    if 'epoch' in log:
        # Mengambil metrik loss
        if 'loss' in log:  # Ini adalah Training Loss
            train_loss.append(log['loss'])
            epochs_loss.append(log['epoch'])
        if 'eval_loss' in log:  # Ini adalah Validation Loss
            val_loss.append(log['eval_loss'])
        
        # Mengambil metrik akurasi (jika Anda menyertakan compute_metrics)
        if 'eval_accuracy' in log:
            val_acc.append(log['eval_accuracy'])

# 2. Plot Grafik Loss (Indikator Utama Overfitting)
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(epochs_loss, train_loss, label='Training Loss', color='#1f77b4', linewidth=2, marker='o')
# Jika panjang val_loss sama dengan epochs_loss (dievaluasi per epoch)
if len(val_loss) == len(epochs_loss):
    plt.plot(epochs_loss, val_loss, label='Validation Loss', color='#ff7f0e', linewidth=2, marker='s')
else:
    # Jika evaluasi hanya di akhir atau interval tertentu
    plt.axhline(y=history[-1].get('eval_loss', 0) if history else 0, color='#ff7f0e', linestyle='--', label='Final Eval Loss')

plt.title('Training vs Validation Loss', fontsize=14, fontweight='bold')
plt.xlabel('Epochs', fontsize=12)
plt.ylabel('Loss Value', fontsize=12)
plt.legend(fontsize=11)

# 3. Interpretasi Visual Langsung di Grafik
plt.subplot(1, 2, 2)
if val_acc:
    plt.plot(range(1, len(val_acc) + 1), val_acc, label='Validation Accuracy', color='#2ca02c', linewidth=2, marker='^')
plt.title('Validation Accuracy Progression', fontsize=14, fontweight='bold')
plt.xlabel('Evaluation Epochs', fontsize=12)
plt.ylabel('Accuracy Score', fontsize=12)
plt.legend(fontsize=11)

plt.tight_layout()
plt.show()

# Validation Evaluation

Evaluasi model pada validation set.

In [ ]:
print("Evaluating on validation set...")

val_predictions = trainer.predict(val_dataset)
val_preds = np.argmax(val_predictions.predictions, axis=-1)
val_labels = np.array(df_val_split["label"].tolist())

# Classification report
target_names = [id2label[i] for i in range(4)]
print("\n=== Validation Classification Report ===")
print(classification_report(val_labels, val_preds, target_names=target_names, digits=4))

# Key metrics
print(f"Accuracy:    {accuracy_score(val_labels, val_preds):.4f}")
print(f"Precision:   {precision_score(val_labels, val_preds, average='macro'):.4f}")
print(f"Recall:      {recall_score(val_labels, val_preds, average='macro'):.4f}")
print(f"F1 Macro:    {f1_score(val_labels, val_preds, average='macro'):.4f}")
print(f"F1 Weighted: {f1_score(val_labels, val_preds, average='weighted'):.4f}")

# Confusion matrix
val_cm = confusion_matrix(val_labels, val_preds)
print("\n=== Validation Confusion Matrix ===")
print(val_cm)

# Test Evaluation

Evaluasi model pada test set. Jika kolom Stance tersedia, evaluasi lengkap akan dilakukan. Jika tidak, hanya distribusi prediksi yang ditampilkan.

In [ ]:
print("Evaluating on test set...")

test_predictions = trainer.predict(test_dataset)
test_preds = np.argmax(test_predictions.predictions, axis=-1)
test_pred_labels = [id2label[p] for p in test_preds]

# Check if test set has labels
if "Stance" in df_test.columns:
    df_test["label"] = df_test["Stance"].map(label2id)
    test_labels = np.array(df_test["label"].tolist())

    target_names = [id2label[i] for i in range(4)]
    print("\n=== Test Classification Report ===")
    print(classification_report(test_labels, test_preds, target_names=target_names, digits=4))

    print(f"Accuracy:    {accuracy_score(test_labels, test_preds):.4f}")
    print(f"Precision:   {precision_score(test_labels, test_preds, average='macro'):.4f}")
    print(f"Recall:      {recall_score(test_labels, test_preds, average='macro'):.4f}")
    print(f"F1 Macro:    {f1_score(test_labels, test_preds, average='macro'):.4f}")
    print(f"F1 Weighted: {f1_score(test_labels, test_preds, average='weighted'):.4f}")

    test_cm = confusion_matrix(test_labels, test_preds)
    print("\n=== Test Confusion Matrix ===")
    print(test_cm)
else:
    print("Test set does not have Stance labels. Showing prediction distribution only.")
    print(pd.Series(test_pred_labels).value_counts())

# Confusion Matrix Visualization

Visualisasi confusion matrix menggunakan seaborn heatmap.

In [ ]:
def plot_confusion_matrix(cm, labels, title="Confusion Matrix"):
    """
    Plot confusion matrix as a seaborn heatmap.

    Args:
        cm: Confusion matrix array
        labels: List of label names
        title: Plot title
    """
    plt.figure(figsize=(8, 6))
    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        cmap="Blues",
        xticklabels=labels,
        yticklabels=labels,
    )
    plt.title(title)
    plt.ylabel("True Label")
    plt.xlabel("Predicted Label")
    plt.tight_layout()
    plt.show()

target_names = [id2label[i] for i in range(4)]

# Validation confusion matrix
plot_confusion_matrix(val_cm, target_names, title="RoBERTa — Validation Confusion Matrix")

# Test confusion matrix (if labels available)
if "Stance" in df_test.columns:
    plot_confusion_matrix(test_cm, target_names, title="RoBERTa — Test Confusion Matrix")

# Error Analysis

Menampilkan 20 sampel yang salah diklasifikasi untuk analisis kesalahan model.

In [ ]:
def error_analysis(df, true_labels, pred_labels, id2label, n=20):
    """
    Display top N misclassified samples.

    Args:
        df: Original dataframe
        true_labels: Array of true label ids
        pred_labels: Array of predicted label ids
        id2label: Mapping from id to label name
        n: Number of samples to display
    """
    misclassified = np.where(true_labels != pred_labels)[0]
    print(f"Total misclassified: {len(misclassified)} / {len(true_labels)} "
          f"({len(misclassified) / len(true_labels) * 100:.2f}%)")

    sample_indices = misclassified[:n]
    print(f"\nShowing top {len(sample_indices)} misclassified samples:\n")

    for rank, idx in enumerate(sample_indices, 1):
        headline = str(df.iloc[idx]["Headline"])
        body = str(df.iloc[idx]["articleBody"])[:200]
        true = id2label[true_labels[idx]]
        pred = id2label[pred_labels[idx]]

        print(f"--- Sample {rank} (index {idx}) ---")
        print(f"True: {true:>10s}  |  Predicted: {pred:>10s}")
        print(f"Headline: {headline}")
        print(f"Article:  {body}...")
        print()

# Run error analysis on validation set
print("=" * 60)
print("ERROR ANALYSIS — Top 20 Misclassified Samples (Validation)")
print("=" * 60)
error_analysis(df_val_split, val_labels, val_preds, id2label, n=20)

# Model Saving

Menyimpan model, tokenizer, dan konfigurasi label ke output directory.

In [ ]:
save_dir = Path(OUTPUT_DIR)
save_dir.mkdir(parents=True, exist_ok=True)

# Save model and tokenizer
model.save_pretrained(save_dir)
tokenizer.save_pretrained(save_dir)

# Save label mappings
label_config = {
    "label2id": label2id,
    "id2label": {str(k): v for k, v in id2label.items()},
}
with open(save_dir / "label_config.json", "w") as f:
    json.dump(label_config, f, indent=2)

# Save training configuration summary
train_config = {
    "model_name": MODEL_NAME,
    "max_length": MAX_LENGTH,
    "batch_size": training_args.per_device_train_batch_size,
    "learning_rate": training_args.learning_rate,
    "weight_decay": training_args.weight_decay,
    "warmup_ratio": training_args.warmup_ratio,
    "num_epochs": training_args.num_train_epochs,
    "fp16": training_args.fp16,
    "seed": SEED,
    "environment": ENVIRONMENT,
    "split_strategy": "GroupShuffleSplit on Body ID",
}
with open(save_dir / "training_config.json", "w") as f:
    json.dump(train_config, f, indent=2)

print(f"Model saved to: {save_dir}")
print(f"Files saved:")
for f_path in sorted(save_dir.iterdir()):
    print(f"  {f_path.name}")

# Longformer vs RoBERTa Comparison

Tabel perbandingan langsung antara kedua model setelah training selesai.

## Model Architecture

| Property | Longformer | RoBERTa |
|---|---|---|
| Model Name | `allenai/longformer-base-4096` | `roberta-base` |
| Maximum Input Length | 4096 tokens | 512 tokens |
| Attention Mechanism | Local + Global attention | Full self-attention |
| Parameters | ~149M | ~125M |
| Global Attention | ✅ (headline + CLS) | ❌ N/A |

## Performance Comparison

| Metric | Longformer | RoBERTa |
|---|---|---|
| Accuracy | _fill after training_ | _fill after training_ |
| F1 Macro | _fill after training_ | _fill after training_ |
| F1 Weighted | _fill after training_ | _fill after training_ |

## Resource Usage

| Resource | Longformer | RoBERTa |
|---|---|---|
| Training Time | _fill after training_ | _fill after training_ |
| GPU Memory Usage | _fill after training_ | _fill after training_ |
| Batch Size (per device) | 2 | 16 |
| Effective Batch Size | 16 (grad accum=8) | 16 |
| MAX_LENGTH | 1024 | 512 |

## Key Observations

- **Longformer** dapat memproses input yang lebih panjang (hingga 4096 token), sehingga lebih sedikit informasi yang hilang akibat truncation. Namun, lebih lambat dan membutuhkan lebih banyak GPU memory.
- **RoBERTa** lebih cepat dan efisien, tetapi dibatasi pada 512 token. Artikel yang lebih panjang akan di-truncate, yang berpotensi mengurangi performa pada kasus di mana konteks panjang penting.
- Kedua model menggunakan strategi data splitting yang identik (`GroupShuffleSplit` pada Body ID) untuk memastikan perbandingan yang fair tanpa data leakage.

## Analysis Notes

1. Jika RoBERTa mendekati performa Longformer → truncation di 512 token tidak terlalu berpengaruh untuk task ini
2. Jika Longformer jauh lebih baik → konteks panjang penting untuk stance detection
3. Perhatikan trade-off antara performa dan efisiensi (training time, memory)